In [ ]:
# FedGATSage (Fed_GNN, branch v2) on NF-ToN-IoT v1 vs NF-ToN-IoT-FHF (labels from matched ToN-IoT flows).
# Same settings for both: <=1500 rows per Attack class (seed 42), 5 clients, 15 rounds, no payload. CPU.
!git clone -q https://github.com/asfi50/Fed_GNN
%cd /kaggle/working/Fed_GNN
!git checkout -q v2 && git log --oneline -1
!export TORCH=$(python -c "import torch; print(torch.__version__.split('+')[0])") && \
export DEVICE=$(python -c "import torch; print('cu128' if torch.cuda.is_available() else 'cpu')") && \
pip install -q torch-scatter torch-sparse pyg-lib torch-geometric -f https://data.pyg.org/whl/torch-${TORCH}+${DEVICE}.html && \
pip install -q -r requirements.txt

In [ ]:
import os
for v in ["OMP_NUM_THREADS","MKL_NUM_THREADS","OPENBLAS_NUM_THREADS","VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"]:
    os.environ[v] = "4"
import torch
torch.set_num_threads(4)
try: torch.set_num_interop_threads(2)
except RuntimeError: pass
print("cuda:", torch.cuda.is_available())

In [ ]:
import urllib.request, pandas as pd
SRC = {"nftoniot": "<signed link from Kaggle MCP download_dataset>",
       "nftoniot_fhf": "<signed link from Kaggle MCP download_dataset>"}
PER_CLASS, SEED = 1500, 42
os.makedirs("/kaggle/working/sets", exist_ok=True)
for name, url in SRC.items():
    raw = f"/kaggle/working/sets/{name}_full.csv"
    urllib.request.urlretrieve(url, raw)
    df = pd.read_csv(raw)
    bal = (df.groupby("Attack", group_keys=False)
             .apply(lambda g: g.sample(n=min(PER_CLASS, len(g)), random_state=SEED))
             .sample(frac=1, random_state=SEED).reset_index(drop=True))
    bal.to_csv(f"/kaggle/working/sets/{name}_bal1500.csv", index=False)
    os.remove(raw)
    print(name, len(bal)); print(bal.Attack.value_counts().to_string(), "\n")

In [ ]:
import subprocess, json, shutil
res = {}
for name in ["nftoniot", "nftoniot_fhf"]:
    d, o = f"data_{name}", f"results_{name}"
    subprocess.run(f"python preprocess_data.py --input_file /kaggle/working/sets/{name}_bal1500.csv --output_dir {d} --num_clients 5", shell=True, check=True)
    subprocess.run(f"python experiments/fedgatsage_experiment.py --data_dir {d} --num_clients 5 --num_rounds 15 --output_dir {o}", shell=True, check=True)
    for src in [o, "results"]:                       # utils may write to results/ regardless of --output_dir
        if os.path.isdir(src):
            shutil.copytree(src, f"/kaggle/working/out/{name}", dirs_exist_ok=True)
    if os.path.isdir("results"):
        shutil.rmtree("results")
    hits = [os.path.join(r, f) for r, _, fs in os.walk(f"/kaggle/working/out/{name}") for f in fs if f.endswith(".json")]
    print(name, "result files:", hits)
    res[name] = {h: json.load(open(h)) for h in hits}
json.dump(res, open("/kaggle/working/out/summary.json", "w"), indent=2, default=str)
print(json.dumps(res, indent=2, default=str)[:6000])